# 05 Dashboard Prep

This notebook prepares final dashboard-ready tables from the outputs we already created in earlier steps.

## 1. Import libraries and define file paths

We import pandas and define the input files and the folder where the final dashboard-ready tables will be saved.

In [1]:
from pathlib import Path
import pandas as pd

BASE_OUTPUT_DIR = Path("..") / "outputs"
DASHBOARD_DIR = BASE_OUTPUT_DIR / "dashboard_ready"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = BASE_OUTPUT_DIR / "events_cleaned.csv"
FUNNEL_OVERALL_PATH = BASE_OUTPUT_DIR / "funnel_overall.csv"
FUNNEL_MONTHLY_PATH = BASE_OUTPUT_DIR / "funnel_monthly.csv"
FUNNEL_CATEGORY_PATH = BASE_OUTPUT_DIR / "funnel_category.csv"
FUNNEL_BRAND_PATH = BASE_OUTPUT_DIR / "funnel_brand.csv"
COHORT_COUNTS_PATH = BASE_OUTPUT_DIR / "cohort_counts.csv"
COHORT_MATRIX_PATH = BASE_OUTPUT_DIR / "cohort_retention_matrix.csv"
COHORT_PERCENTAGES_PATH = BASE_OUTPUT_DIR / "cohort_retention_percentages.csv"

KPI_SUMMARY_OUTPUT = DASHBOARD_DIR / "kpi_summary.csv"
FUNNEL_SUMMARY_OUTPUT = DASHBOARD_DIR / "funnel_summary.csv"
MONTHLY_TRENDS_OUTPUT = DASHBOARD_DIR / "monthly_trends.csv"
CATEGORY_PERFORMANCE_OUTPUT = DASHBOARD_DIR / "category_performance.csv"
BRAND_PERFORMANCE_OUTPUT = DASHBOARD_DIR / "brand_performance.csv"
COHORT_HEATMAP_OUTPUT = DASHBOARD_DIR / "cohort_heatmap.csv"

print(f"Dashboard output folder: {DASHBOARD_DIR.resolve()}")

Dashboard output folder: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready


## 2. Load the existing project outputs

We load the cleaned data and the summary files created in the earlier notebooks.

In [2]:
events = pd.read_csv(EVENTS_PATH, parse_dates=["event_time"])
funnel_overall = pd.read_csv(FUNNEL_OVERALL_PATH)
funnel_monthly = pd.read_csv(FUNNEL_MONTHLY_PATH)
funnel_category = pd.read_csv(FUNNEL_CATEGORY_PATH)
funnel_brand = pd.read_csv(FUNNEL_BRAND_PATH)
cohort_counts = pd.read_csv(COHORT_COUNTS_PATH)
cohort_matrix = pd.read_csv(COHORT_MATRIX_PATH)
cohort_percentages = pd.read_csv(COHORT_PERCENTAGES_PATH)

print("All input files loaded successfully.")

All input files loaded successfully.


## 3. Prepare a KPI summary table

This table gives a compact overview of the main project KPIs for a dashboard header or summary cards.

In [3]:
overall_row = funnel_overall.iloc[0]

kpi_summary = pd.DataFrame(
    {
        "metric_name": [
            "total_events",
            "unique_users",
            "unique_sessions",
            "total_views",
            "total_carts",
            "total_purchases",
            "view_to_cart_rate",
            "cart_to_purchase_rate",
            "view_to_purchase_rate",
            "date_range_start",
            "date_range_end",
        ],
        "metric_value": [
            len(events),
            events["user_id"].nunique(),
            events["user_session"].nunique(),
            int(overall_row["views"]),
            int(overall_row["carts"]),
            int(overall_row["purchases"]),
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["cart_to_purchase_rate"]), 4),
            round(float(overall_row["view_to_purchase_rate"]), 4),
            events["event_time"].min().strftime("%Y-%m-%d %H:%M:%S"),
            events["event_time"].max().strftime("%Y-%m-%d %H:%M:%S"),
        ],
    }
)

kpi_summary

,metric_name,metric_value
0,total_events,884474
1,unique_users,407283
2,unique_sessions,490399
3,total_views,793099
4,total_carts,54032
5,total_purchases,37343
6,view_to_cart_rate,0.0681
7,cart_to_purchase_rate,0.6911
8,view_to_purchase_rate,0.0471
9,date_range_start,2020-09-24 11:57:06


## 4. Prepare a clean funnel summary table

This table is a tidy version of the overall funnel metrics and is useful for KPI cards or a simple funnel visual.

In [4]:
funnel_summary = pd.DataFrame(
    {
        "funnel_stage": ["view", "cart", "purchase"],
        "event_count": [
            int(overall_row["views"]),
            int(overall_row["carts"]),
            int(overall_row["purchases"]),
        ],
        "conversion_from_previous_stage": [
            1.0,
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["cart_to_purchase_rate"]), 4),
        ],
        "conversion_from_view": [
            1.0,
            round(float(overall_row["view_to_cart_rate"]), 4),
            round(float(overall_row["view_to_purchase_rate"]), 4),
        ],
    }
)

funnel_summary

,funnel_stage,event_count,conversion_from_previous_stage,conversion_from_view
0,view,793099,1.0000,1.0000
1,cart,54032,0.0681,0.0681
2,purchase,37343,0.6911,0.0471


## 5. Prepare monthly trends

This table combines monthly funnel metrics with a few additional monthly activity metrics so it is useful for time-series dashboard charts.

In [5]:
monthly_activity = (
    events.groupby("event_month")
    .agg(
        total_events=("event_type", "size"),
        unique_users=("user_id", "nunique"),
        unique_sessions=("user_session", "nunique"),
        average_price=("price", "mean"),
    )
    .reset_index()
)

monthly_trends = monthly_activity.merge(funnel_monthly, on="event_month", how="left")
monthly_trends = monthly_trends.rename(columns={
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

monthly_trends["average_price"] = monthly_trends["average_price"].round(2)
monthly_trends = monthly_trends.sort_values("event_month")
monthly_trends

,event_month,total_events,unique_users,unique_sessions,average_price,total_views,total_carts,total_purchases,view_to_cart_conversion_rate,cart_to_purchase_conversion_rate,view_to_purchase_conversion_rate
0,2020-09,28059,15334,16991,117.38,25640,1412,1007,0.055070,0.713173,0.039275
1,2020-10,161418,84216,95767,113.70,146414,8728,6276,0.059612,0.719065,0.042865
2,2020-11,188091,92600,108579,123.15,170082,10435,7574,0.061353,0.725827,0.044531
3,2020-12,152590,72137,85661,140.28,136719,9342,6529,0.068330,0.698887,0.047755
4,2021-01,187441,81256,98308,180.72,166431,12695,8315,0.076278,0.654982,0.049961
5,2021-02,166875,74606,89011,175.80,147813,11420,7642,0.077260,0.669177,0.051700


## 6. Prepare category performance

This table keeps the most useful category-level funnel metrics and limits the output to stronger categories for easier dashboarding.

In [6]:
category_performance = funnel_category.copy()
category_performance = category_performance.rename(columns={
    "category_code": "category_name",
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

category_performance = category_performance[category_performance["total_views"] >= 100].copy()
category_performance = category_performance.sort_values(
    ["view_to_purchase_conversion_rate", "total_purchases"],
    ascending=[False, False],
).head(25)
category_performance

,category_name,total_views,total_carts,total_purchases,view_to_cart_conversion_rate,cart_to_purchase_conversion_rate,view_to_purchase_conversion_rate
0,computers.peripherals.camera,4363,589,521,0.134999,0.884550,0.119413
1,computers.peripherals.scanner,1599,200,165,0.125078,0.825000,0.103189
2,stationery.stapler,741,48,66,0.064777,1.375000,0.089069
3,stationery.cartrige,32919,3041,2739,0.092378,0.900691,0.083204
4,computers.ebooks,2825,268,212,0.094867,0.791045,0.075044
5,computers.components.videocards,97039,12684,6888,0.130710,0.543046,0.070982
6,electronics.video.projector,1374,118,94,0.085881,0.796610,0.068413
7,computers.peripherals.printer,37453,3183,2557,0.084987,0.803330,0.068272
8,electronics.video.tv_remote,932,68,62,0.072961,0.911765,0.066524
9,electronics.calculator,152,8,10,0.052632,1.250000,0.065789


## 7. Prepare brand performance

This table keeps the most useful brand-level funnel metrics and limits the output to stronger brands for easier dashboarding.

In [7]:
brand_performance = funnel_brand.copy()
brand_performance = brand_performance.rename(columns={
    "brand": "brand_name",
    "views": "total_views",
    "carts": "total_carts",
    "purchases": "total_purchases",
    "view_to_cart_rate": "view_to_cart_conversion_rate",
    "cart_to_purchase_rate": "cart_to_purchase_conversion_rate",
    "view_to_purchase_rate": "view_to_purchase_conversion_rate",
})

brand_performance = brand_performance[brand_performance["total_views"] >= 100].copy()
brand_performance = brand_performance.sort_values(
    ["view_to_purchase_conversion_rate", "total_purchases"],
    ascending=[False, False],
).head(25)
brand_performance

,brand_name,total_views,total_carts,total_purchases,view_to_cart_conversion_rate,cart_to_purchase_conversion_rate,view_to_purchase_conversion_rate
9,codyson,365,63,71,0.172603,1.126984,0.194521
12,pmma,127,24,21,0.188976,0.875000,0.165354
18,nv-print,2825,401,416,0.141947,1.037406,0.147257
19,content,166,29,24,0.174699,0.827586,0.144578
24,keenetic,701,91,94,0.129815,1.032967,0.134094
25,saitek,750,118,97,0.157333,0.822034,0.129333
26,p.i.t,604,65,77,0.107616,1.184615,0.127483
30,fluke,121,11,15,0.090909,1.363636,0.123967
32,sapphire,8019,1440,917,0.179574,0.636806,0.114353
37,canyon,633,64,70,0.101106,1.093750,0.110585


## 8. Prepare the cohort heatmap table

We reshape the retention percentage table into a tidy format so it can be used easily in dashboard tools.

In [8]:
cohort_heatmap = cohort_percentages.rename(columns={cohort_percentages.columns[0]: "cohort_month"}).copy()
cohort_heatmap = cohort_heatmap.melt(
    id_vars="cohort_month",
    var_name="months_since_first_event",
    value_name="retention_rate",
)
cohort_heatmap["months_since_first_event"] = pd.to_numeric(cohort_heatmap["months_since_first_event"], errors="coerce")
cohort_heatmap = cohort_heatmap.sort_values(["cohort_month", "months_since_first_event"])
cohort_heatmap

,cohort_month,months_since_first_event,retention_rate
0,2020-09,0,1.000000
6,2020-09,1,0.062280
12,2020-09,2,0.017412
18,2020-09,3,0.007761
24,2020-09,4,0.006391
30,2020-09,5,0.004695
1,2020-10,0,1.000000
7,2020-10,1,0.026747
13,2020-10,2,0.008143
19,2020-10,3,0.005333


## 9. Save the dashboard-ready tables

These are the final files that a dashboard tool can use directly.

In [9]:
kpi_summary.to_csv(KPI_SUMMARY_OUTPUT, index=False)
funnel_summary.to_csv(FUNNEL_SUMMARY_OUTPUT, index=False)
monthly_trends.to_csv(MONTHLY_TRENDS_OUTPUT, index=False)
category_performance.to_csv(CATEGORY_PERFORMANCE_OUTPUT, index=False)
brand_performance.to_csv(BRAND_PERFORMANCE_OUTPUT, index=False)
cohort_heatmap.to_csv(COHORT_HEATMAP_OUTPUT, index=False)

print(f"Saved: {KPI_SUMMARY_OUTPUT.resolve()}")
print(f"Saved: {FUNNEL_SUMMARY_OUTPUT.resolve()}")
print(f"Saved: {MONTHLY_TRENDS_OUTPUT.resolve()}")
print(f"Saved: {CATEGORY_PERFORMANCE_OUTPUT.resolve()}")
print(f"Saved: {BRAND_PERFORMANCE_OUTPUT.resolve()}")
print(f"Saved: {COHORT_HEATMAP_OUTPUT.resolve()}")

Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\kpi_summary.csv
Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\funnel_summary.csv
Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\monthly_trends.csv
Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\category_performance.csv
Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\brand_performance.csv
Saved: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready\cohort_heatmap.csv


## 10. Print a short summary of the dashboard-ready tables

This final summary explains which files were created and what each one is for.

In [10]:
table_summary = pd.DataFrame(
    {
        "file_name": [
            "kpi_summary.csv",
            "funnel_summary.csv",
            "monthly_trends.csv",
            "category_performance.csv",
            "brand_performance.csv",
            "cohort_heatmap.csv",
        ],
        "purpose": [
            "High-level KPI cards and summary metrics",
            "Overall funnel counts and conversion rates",
            "Monthly trend charts and time-series analysis",
            "Category-level funnel performance for ranking and comparison",
            "Brand-level funnel performance for ranking and comparison",
            "Cohort retention heatmap source table",
        ],
    }
)

print("Dashboard-ready tables created:")
for _, row in table_summary.iterrows():
    print(f"- {row['file_name']}: {row['purpose']}")

print(f"\nOutput folder: {DASHBOARD_DIR.resolve()}")

Dashboard-ready tables created:
- kpi_summary.csv: High-level KPI cards and summary metrics
- funnel_summary.csv: Overall funnel counts and conversion rates
- monthly_trends.csv: Monthly trend charts and time-series analysis
- category_performance.csv: Category-level funnel performance for ranking and comparison
- brand_performance.csv: Brand-level funnel performance for ranking and comparison
- cohort_heatmap.csv: Cohort retention heatmap source table

Output folder: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\dashboard_ready
